# 07. 테이블 파운데이션 모델 비교 (TabPFN vs TabICL vs TabDPT)

TabPFN처럼 **미리 학습된(in-context learning) 테이블 파운데이션 모델**끼리, **06 최종 조합과 같은 변수 · 같은 CV**로 비교한다.

| 모델 | 만든 곳 | 패키지 |
| --- | --- | --- |
| TabPFN v3.5 | Prior Labs | `tabpfn` (06 결과 재사용) |
| TabICL v2 | Inria (soda-inria) | `tabicl` |
| TabDPT | Layer 6 | `tabdpt` |

- **변수**: 06 최종 = 베이스라인 + 데뷔작(결측 그대로) + 배급사 정리 + 개봉 시기 + 관람 등급 (41개) · **sqrt 타깃**
- **검증**: 팀 공통 5-Fold (seed 42), 원래 관객수 단위 RMSE · 결과는 `cache_07/`에 저장
- 세 모델 모두 결측(NaN)을 직접 처리함

> **Mitra (Amazon)는 제외**: AutoGluon에 포함돼 있는데 설치하면 pandas · torch가 구버전으로 바뀌고, 이 PC의 윈도우 앱 제어 정책이 새로 받은 torch DLL을 차단해서 실행 불가. 필요하면 코랩에서 따로 실행.
> **설치** (처음 한 번): `%pip install tabicl` · `%pip install --no-deps tabdpt` · `%pip install faiss-cpu omegaconf`
> (`tabdpt`를 그냥 설치하면 TabPFN이 쓰는 `huggingface_hub`가 구버전으로 바뀌어서 `--no-deps`로 설치)

## 0. 설정

In [1]:
import os
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn import set_config
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from tabicl import TabICLRegressor
from tabdpt import TabDPTRegressor

from features import make_features

warnings.filterwarnings('ignore')
set_config(display='text')

DATA_DIR = '.'
SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE_DIR = 'cache_07'
os.makedirs(CACHE_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, 'movies_train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'movies_test.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
y = train['box_off_num']
hit = y >= 1_000_000

fwd, inv = np.sqrt, lambda p: np.square(np.clip(p, 0, None))
OPTS = {'debut': 1, 'debut_mode': 'nan', 'distributor': 1, 'date': 1, 'rating': 1}   # 06 최종
X, X_test = make_features(train, test, OPTS)
print('device:', DEVICE, '| 변수', X.shape[1], '개 | 결측', int(X.isna().sum().sum()), '개')

C:\Users\taehe\OneDrive\문서\대학교\3학년 2학기\ESAA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu | 변수 41 개 | 결측 330 개


## 1. 모델 정의
- 각 모델은 scikit-learn처럼 `fit` / `predict`
- TabDPT는 numpy 배열 입력, CPU에서는 `compile` · `flash attention` 끔

In [2]:
class TabDPTWrapper:
    """TabDPT를 다른 모델과 같은 방식(fit/predict, DataFrame 입력)으로 쓰기 위한 래퍼"""
    def __init__(self, seed=SEED):
        self.seed = seed
        self.model = TabDPTRegressor(device=DEVICE, compile=False, use_flash=(DEVICE == 'cuda'), verbose=False)

    def fit(self, X, y):
        self.model.fit(np.asarray(X, dtype=float), np.asarray(y, dtype=float))
        return self

    def predict(self, X):
        return self.model.predict(np.asarray(X, dtype=float), seed=self.seed)


MODELS = {
    'TabPFN v3.5': lambda: TabPFNRegressor.create_default_for_version(ModelVersion.V3_5, device=DEVICE, random_state=SEED),
    'TabICL':      lambda: TabICLRegressor(device=DEVICE, random_state=SEED),
    'TabDPT':      lambda: TabDPTWrapper(),
}

## 2. 모델별 CV

In [3]:
def run_cv(name):
    path = os.path.join(CACHE_DIR, f"{name.replace(' ', '_')}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    # TabPFN v3.5는 06에서 같은 조건으로 돌린 결과 재사용
    if name == 'TabPFN v3.5' and os.path.exists('cache_06/M_nan.pkl'):
        r = pickle.load(open('cache_06/M_nan.pkl', 'rb'))
        res = {'모델': name, 'CV RMSE': r['CV RMSE'], 'fold 표준편차': r['fold 표준편차'],
               '흥행작 RMSE': r['흥행작 RMSE'], '흥행작 외 RMSE': mean_squared_error(y[~hit], r['oof'][~hit]) ** 0.5,
               '소요(초)': r['소요(초)'], 'oof': r['oof']}
        pickle.dump(res, open(path, 'wb'))
        return res
    oof = np.zeros(len(y))
    fold_rmse = []
    t0 = time.time()
    for tr_idx, va_idx in kf.split(X):
        model = MODELS[name]()
        model.fit(X.iloc[tr_idx], fwd(y.iloc[tr_idx]))
        oof[va_idx] = np.clip(inv(model.predict(X.iloc[va_idx])), 0, None)
        fold_rmse.append(mean_squared_error(y.iloc[va_idx], oof[va_idx]) ** 0.5)
    res = {'모델': name, 'CV RMSE': mean_squared_error(y, oof) ** 0.5, 'fold 표준편차': np.std(fold_rmse),
           '흥행작 RMSE': mean_squared_error(y[hit], oof[hit]) ** 0.5,
           '흥행작 외 RMSE': mean_squared_error(y[~hit], oof[~hit]) ** 0.5,
           '소요(초)': time.time() - t0, 'oof': oof}
    pickle.dump(res, open(path, 'wb'))
    return res


results = {}
for name in MODELS:
    try:
        results[name] = r = run_cv(name)
        print(f"{name:<12} CV RMSE {r['CV RMSE']:>11,.0f} | fold std {r['fold 표준편차']:>9,.0f} | 흥행작 {r['흥행작 RMSE']:>11,.0f} | {r['소요(초)']:.0f}초")
    except Exception as e:
        print(f'{name:<12} 실패: {type(e).__name__}: {str(e)[:200]}')

TabPFN v3.5  CV RMSE   1,350,295 | fold std   314,851 | 흥행작   3,025,307 | 365초


TabICL       CV RMSE   1,380,841 | fold std   315,311 | 흥행작   3,135,799 | 213초


TabDPT       CV RMSE   1,391,539 | fold std   339,352 | 흥행작   3,146,137 | 198초


In [4]:
FMT = {'CV RMSE': '{:,.0f}', 'fold 표준편차': '{:,.0f}', '흥행작 RMSE': '{:,.0f}', '흥행작 외 RMSE': '{:,.0f}', '소요(초)': '{:.0f}'}
table = pd.DataFrame([{k: v for k, v in r.items() if k != 'oof'} for r in results.values()]).sort_values('CV RMSE')
display(table.style.format(FMT).highlight_min(subset=['CV RMSE', '흥행작 RMSE', '흥행작 외 RMSE'], color='#c6efce'))
best_name = table.iloc[0]['모델']
print('→ CV 최고 모델:', best_name)

,모델,CV RMSE,fold 표준편차,흥행작 RMSE,흥행작 외 RMSE,소요(초)
0,TabPFN v3.5,"1,350,295","314,851","3,025,307","442,997",365
1,TabICL,"1,380,841","315,311","3,135,799","383,493",213
2,TabDPT,"1,391,539","339,352","3,146,137","410,966",198


→ CV 최고 모델: TabPFN v3.5


## 3. 예측이 얼마나 비슷한지
상관이 낮으면 서로 다른 실수를 한다는 뜻 → 평균내면 좋아질 여지가 있음

In [5]:
oofs = pd.DataFrame({n: r['oof'] for n, r in results.items()})
display(oofs.corr().style.format('{:.3f}').background_gradient(cmap='Blues'))

top = y.sort_values(ascending=False).index[:10]
cmp = pd.concat([train.loc[top, ['title']], y[top].rename('실제'), oofs.loc[top]], axis=1)
display(cmp.style.format({c: '{:,.0f}' for c in cmp.columns if c != 'title'}))

,TabPFN v3.5,TabICL,TabDPT
TabPFN v3.5,1.000,0.986,0.984
TabICL,0.986,1.000,0.984
TabDPT,0.984,0.984,1.000


,title,실제,TabPFN v3.5,TabICL,TabDPT
372,국제시장,"14,262,766","3,271,494","2,963,304","3,422,525"
362,도둑들,"12,983,841","8,438,953","6,151,680","7,136,792"
530,7번방의 선물,"12,811,435","4,771,265","3,546,803","3,528,670"
498,암살,"12,706,663","6,364,228","5,742,076","5,948,379"
460,"광해, 왕이 된 남자","12,323,595","2,016,706","2,100,257","1,947,011"
122,변호인,"11,374,879","2,311,562","2,697,208","2,286,336"
496,설국열차,"9,350,351","959,755","479,191","604,931"
101,관상,"9,135,806","3,363,345","3,646,398","3,671,903"
505,해적: 바다로 간 산적,"8,666,208","3,382,569","3,222,016","3,882,516"
476,수상한 그녀,"8,659,725","4,336,170","3,991,104","3,702,436"


## 4. (참고) 파운데이션 모델끼리 예측 평균
모두 파운데이션 모델이라 "TabPFN류만 사용" 조건 안에서의 평균. 1% 이상 좋아질 때만 채택.

In [6]:
from itertools import combinations
rows = []
names = list(results)
for k in range(1, len(names) + 1):
    for combo in combinations(names, k):
        avg = np.mean([results[n]['oof'] for n in combo], axis=0)
        rows.append({'조합': ' + '.join(combo), 'CV RMSE': mean_squared_error(y, avg) ** 0.5,
                     '흥행작 RMSE': mean_squared_error(y[hit], avg[hit]) ** 0.5})
avg_df = pd.DataFrame(rows).sort_values('CV RMSE')
display(avg_df.style.format({'CV RMSE': '{:,.0f}', '흥행작 RMSE': '{:,.0f}'}).highlight_min(subset=['CV RMSE'], color='#c6efce'))

best_single = table.iloc[0]['CV RMSE']
best_avg = avg_df.iloc[0]
USE_AVG = (' + ' in best_avg['조합']) and best_avg['CV RMSE'] < best_single * 0.99
print(f"단일 최고 {best_single:,.0f} vs 평균 최고 {best_avg['CV RMSE']:,.0f} ({best_avg['조합']}) → 평균 사용: {USE_AVG}")

,조합,CV RMSE,흥행작 RMSE
0,TabPFN v3.5,"1,350,295","3,025,307"
3,TabPFN v3.5 + TabICL,"1,359,696","3,068,529"
4,TabPFN v3.5 + TabDPT,"1,367,095","3,078,363"
6,TabPFN v3.5 + TabICL + TabDPT,"1,367,820","3,089,223"
1,TabICL,"1,380,841","3,135,799"
5,TabICL + TabDPT,"1,381,476","3,130,688"
2,TabDPT,"1,391,539","3,146,137"


단일 최고 1,350,295 vs 평균 최고 1,350,295 (TabPFN v3.5) → 평균 사용: False


## 5. 제출 파일

In [7]:
def fit_predict(name):
    model = MODELS[name]()
    model.fit(X, fwd(y))
    return np.clip(inv(model.predict(X_test)), 0, None)


def save(pred, fname):
    sub = submission.copy()
    sub['box_off_num'] = pred
    sub.to_csv(fname, index=False)
    print(f'저장: {fname} | 예측 평균 {pred.mean():,.0f} · 최대 {pred.max():,.0f}')


# TabPFN v3.5 제출 파일은 06_M_nan.csv와 같으므로, 다른 모델만 저장
test_preds = {}
for name in results:
    if name == 'TabPFN v3.5':
        continue
    test_preds[name] = fit_predict(name)
    save(test_preds[name], f"07_{name.replace(' ', '_')}_sqrt.csv")

if USE_AVG:
    combo = best_avg['조합'].split(' + ')
    if 'TabPFN v3.5' in combo:
        test_preds['TabPFN v3.5'] = fit_predict('TabPFN v3.5')
    save(np.mean([test_preds[n] for n in combo], axis=0), '07_avg_' + '+'.join(n.replace(' ', '') for n in combo) + '.csv')

table.to_csv('07_results.csv', index=False, encoding='utf-8-sig')
print('07_results.csv 저장 (노션 기록용)')

저장: 07_TabICL_sqrt.csv | 예측 평균 539,228 · 최대 7,299,748


저장: 07_TabDPT_sqrt.csv | 예측 평균 574,923 · 최대 7,234,938
07_results.csv 저장 (노션 기록용)
